# NLP EXAM QUICK REFERENCE
**Find any solution in <30 seconds**

---

## MIDTERM (80 min)

---

### Task 1: Text Processing
**See:** `01_Text_Processing_Template.ipynb`

**Key steps:** Read file → Clean (remove punctuation) → Lowercase → Extract emails → Tokenize → FreqDist → Plot CFD

```python
# Quick copy-paste solution:
import re, string, nltk
from nltk import FreqDist

# Read
with open('file.txt', 'r') as f: text = f.read()

# Extract emails FIRST
emails = re.findall(r'[\w\.]+@[\w\.]+', text)

# Clean
clean = text.translate(str.maketrans('', '', string.punctuation)).lower()

# Tokenize
tokens = [t for t in nltk.word_tokenize(clean) if t.isalpha()]

# Frequency
fdist = FreqDist(tokens)
fdist.most_common(10)

# CFD (by first letter)
cfd = nltk.ConditionalFreqDist((w[0], len(w)) for w in tokens if w[0] in 'abc')
cfd.plot()
```

---

### Task 2: Corpus + WordNet
**See:** `02_NLTK_Corpus_Operations.ipynb`

**Key steps:** Load Gutenberg → Get words → FreqDist → Most common word → WordNet synonyms/antonyms

```python
from nltk.corpus import gutenberg, wordnet as wn
from nltk import FreqDist

# Gutenberg most common word
words = gutenberg.words('austen-persuasion.txt')
fdist = FreqDist(words)
most_common = fdist.max()

# WordNet synonyms
syns = [lemma.name() for s in wn.synsets('happy') for lemma in s.lemmas()]

# WordNet antonyms
ants = [ant.name() for s in wn.synsets('happy') 
        for lemma in s.lemmas() for ant in lemma.antonyms()]
```

---

### Task 3: POS Tagging (Bigram)
**See:** `03_POS_Tagging_Chunking.ipynb` (Part A)

**Key steps:** Get Brown tagged_sents → Split train/test → Train Bigram (with backoff) → Evaluate accuracy

```python
from nltk.corpus import brown
from nltk.tag import UnigramTagger, BigramTagger, DefaultTagger

# Get data
tagged_sents = brown.tagged_sents(tagset='universal')

# Split
size = int(len(tagged_sents) * 0.9)
train_sents = tagged_sents[:size]
test_sents = tagged_sents[size:]

# Train with backoff chain
t0 = DefaultTagger('NOUN')
t1 = UnigramTagger(train_sents, backoff=t0)
t2 = BigramTagger(train_sents, backoff=t1)

# Evaluate
accuracy = t2.accuracy(test_sents)
print(f'Accuracy: {accuracy:.4f}')
```

---

## FINAL (45 min)

---

### Task 1: Feature Extraction (TF-IDF/BoW)
**See:** `04_ML_Classification.ipynb` (Patterns 1-2)

**Key steps:** Create vectorizer → fit_transform on texts → Display features/matrix

```python
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

texts = ["text 1", "text 2", "text 3"]

# TF-IDF
tfidf = TfidfVectorizer(lowercase=True, stop_words='english')
X = tfidf.fit_transform(texts)
print('Features:', tfidf.get_feature_names_out())
print('Matrix:', X.toarray())

# Bag of Words
bow = CountVectorizer(lowercase=True, stop_words='english')
X_bow = bow.fit_transform(texts)
```

---

### Task 2: Classification (Naive Bayes)
**See:** `04_ML_Classification.ipynb` (Patterns 3-6)

**Key steps:** Split data → Vectorize → Train NB → Predict → Evaluate

```python
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42, stratify=labels
)

# Pipeline (recommended)
model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english')),
    ('clf', MultinomialNB())
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Evaluate
print(f'Accuracy: {accuracy_score(y_test, predictions):.4f}')
print(classification_report(y_test, predictions))
```

---

### Task 3: Chunking (NP Extraction)
**See:** `03_POS_Tagging_Chunking.ipynb` (Part B)

**Key steps:** Define grammar → Create parser → POS tag sentence → Parse → Extract NPs

```python
import nltk

# Grammar for NP
grammar = "NP: {<DT>?<JJ>*<NN.*>+}"
cp = nltk.RegexpParser(grammar)

# From text
text = "The quick brown fox jumps over the lazy dog."
tokens = nltk.word_tokenize(text)
tagged = nltk.pos_tag(tokens)
tree = cp.parse(tagged)

# Extract NPs
for subtree in tree.subtrees():
    if subtree.label() == 'NP':
        print(' '.join(word for word, tag in subtree.leaves()))
```

---

## GRAMMAR PATTERNS CHEATSHEET

| Pattern | Grammar | Example |
|---------|---------|--------|
| Basic NP | `{<DT>?<JJ>*<NN>}` | "the big dog" |
| NP + Proper | `{<DT>?<JJ>*<NN>}` `{<NNP>+}` | "John Smith" |
| Any NN | `{<DT>?<JJ>*<NN.*>+}` | "the quick dogs" |

---

## POS TAGS CHEATSHEET

| Tag | Meaning | Example |
|-----|---------|--------|
| NN | Noun | dog |
| NNS | Noun plural | dogs |
| NNP | Proper noun | John |
| VB | Verb base | run |
| VBD | Verb past | ran |
| JJ | Adjective | big |
| DT | Determiner | the, a |
| IN | Preposition | in, on |

---

## KEY FUNCTIONS CHEATSHEET

| Task | Function |
|------|----------|
| Read file | `with open(f,'r') as f: text = f.read()` |
| Remove punct | `text.translate(str.maketrans('','',string.punctuation))` |
| Lowercase | `text.lower()` |
| Find emails | `re.findall(r'[\w\.]+@[\w\.]+', text)` |
| Tokenize | `nltk.word_tokenize(text)` |
| FreqDist | `FreqDist(tokens).most_common(10)` |
| Tagged sents | `brown.tagged_sents(tagset='universal')` |
| Synonyms | `[l.name() for s in wn.synsets('word') for l in s.lemmas()]` |
| Antonyms | `[a.name() for s in wn.synsets('word') for l in s.lemmas() for a in l.antonyms()]` |
| TF-IDF | `TfidfVectorizer().fit_transform(texts)` |
| Split | `train_test_split(X, y, test_size=0.3)` |
| Naive Bayes | `MultinomialNB().fit(X, y)` |
| Accuracy | `accuracy_score(y_test, predictions)` |

---

## NOTEBOOK INDEX

| Notebook | Contents | Exam |
|----------|----------|------|
| `00_Quick_Imports.ipynb` | All imports + downloads | Both |
| `01_Text_Processing_Template.ipynb` | File I/O, cleaning, regex, tokenization, FreqDist, CFD | Midterm |
| `02_NLTK_Corpus_Operations.ipynb` | Gutenberg, Brown, WordNet | Midterm |
| `03_POS_Tagging_Chunking.ipynb` | Bigram tagger, chunking, NP extraction | Both |
| `04_ML_Classification.ipynb` | TF-IDF, train/test, Naive Bayes, evaluation | Final |